In [ ]:
!pip install dowhy
!pip install networkx

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 403.1/403.1 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 245.5/245.5 kB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.4/4.4 MB 55.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.3/37.3 MB 29.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 30.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.1/12.1 MB 43.5 MB/s eta 0:00:00
  Attempting uninstall: scipy
    Found existing installation: scipy 1.16.3
    Uninstalling scipy-1.16.3:
      Successfully uninstalled scipy-1.16.3
  Attempting uninstall: cvxpy
    Found existing installation: cvxpy 1.6.7
    Uninstalling cvxpy-1.6.7:
      Successfully uninstalled cvxpy-1.6.7


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
project_path = "/content/drive/MyDrive/Colab Notebooks/CD Skripsi"

In [ ]:
import networkx as nx
from dowhy import gcm
import pandas as pd
import numpy as np

In [ ]:
# Set seed
np.random.seed(42)

In [ ]:
data = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/CD Skripsi/EdgesDAG - Cluster 1_PD2.5.csv')

data

,No.,Node 1,Interaction,Node 2,Ensemble,Edge,No edge,--> dd pl,<-- dd pl,---,--> pd nl,<-- pd nl,--> dd nl,<-- dd nl,o->,<-o,o-o,<->
0,1,delivery_time,-->,is_dissatisfied,1.000,1.000,NaN,NaN,NaN,NaN,NaN,NaN,1.000,NaN,NaN,NaN,NaN,NaN
1,2,freight_value,-->,delivery_time,1.000,1.000,NaN,NaN,NaN,NaN,NaN,NaN,1.000,NaN,NaN,NaN,NaN,NaN
2,3,avg_installments,-->,total_payment,1.000,1.000,NaN,1.000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,4,delivery_delay,-->,late_delivery_flag,1.000,1.000,NaN,1.000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,5,delivery_time,-->,late_delivery_flag,1.000,1.000,NaN,NaN,NaN,NaN,1.000,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,6,freight_value,-->,delivery_delay,1.000,1.000,NaN,NaN,NaN,NaN,1.000,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,7,late_delivery_flag,-->,is_dissatisfied,1.000,1.000,NaN,NaN,NaN,NaN,1.000,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,8,product_description_length,-->,total_payment,1.000,1.000,NaN,1.000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,9,total_payment,-->,delivery_time,1.000,1.000,NaN,NaN,NaN,NaN,1.000,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,10,product_description_length,---,product_photos_qty,1.000,1.000,NaN,NaN,NaN,1.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


#DoWhy

In [ ]:
data_cluster1 = pd.read_csv(
    '/content/drive/MyDrive/Colab Notebooks/CD Skripsi/causal_cluster_1.csv',
    sep=',',
    engine='python'
)

data_cluster1.head()

,cancellation,no_delivery,delivery_delay,delivery_time,late_delivery_flag,avg_installments,total_payment,freight_value,product_description_length,product_photos_qty,is_dissatisfied
0,0.0,0.0,-5.0,6.0,0.0,8.0,141.90,12.00,236.0,1.0,0
1,0.0,0.0,-5.0,3.0,0.0,1.0,27.19,8.29,635.0,1.0,0
2,0.0,0.0,-12.0,1.0,0.0,8.0,166.98,12.98,360.0,3.0,0
3,0.0,0.0,-12.0,6.0,0.0,1.0,35.38,7.39,818.0,6.0,0
4,0.0,0.0,31.0,53.0,1.0,4.0,129.76,24.86,1023.0,1.0,1


In [ ]:
graph = nx.DiGraph()
graph.add_edges_from([
    ('delivery_time', 'is_dissatisfied'),
        ('freight_value', 'delivery_time'),
        ('avg_installments', 'total_payment'),
        ('delivery_delay', 'late_delivery_flag'),
        ('delivery_time', 'late_delivery_flag'),
        ('freight_value', 'delivery_delay'),
        ('late_delivery_flag', 'is_dissatisfied'),
        ('product_description_length', 'total_payment'),
        ('total_payment', 'delivery_time'),

        ('avg_installments', 'freight_value'),
        ('freight_value', 'is_dissatisfied'),
        ('freight_value', 'late_delivery_flag'),
        ('total_payment', 'no_delivery'),
        ('delivery_delay', 'delivery_time'),
        ('freight_value', 'no_delivery'),
        ('product_photos_qty', 'freight_value'),
        ('no_delivery', 'cancellation'),
        ('product_description_length', 'freight_value'),
        ('product_photos_qty', 'delivery_time'),
        ('no_delivery', 'is_dissatisfied'),
        ('avg_installments', 'no_delivery'),
        ('delivery_time', 'no_delivery'),
        ('delivery_delay', 'no_delivery'),
        ('freight_value', 'cancellation'),
        ('total_payment', 'cancellation')
])

In [ ]:
causal_model = gcm.StructuralCausalModel(graph)

In [ ]:
gcm.auto.assign_causal_mechanisms(causal_model, data_cluster1)

In [ ]:
gcm.fit(causal_model, data_cluster1)

Fitting causal mechanism of node cancellation: 100%|██████████| 11/11 [00:02<00:00,  3.82it/s]


In [ ]:
validation_summary = gcm.evaluate_causal_model(causal_model, data_cluster1)
validation_summary

Test permutations of given graph: 100%|██████████| 50/50 [05:08<00:00,  6.16s/it]


CausalModelEvaluationResult(mechanism_performances={'avg_installments': MechanismPerformanceResult(), 'product_description_length': MechanismPerformanceResult(), 'product_photos_qty': MechanismPerformanceResult(), 'total_payment': MechanismPerformanceResult(), 'freight_value': MechanismPerformanceResult(), 'delivery_delay': MechanismPerformanceResult(), 'delivery_time': MechanismPerformanceResult(), 'late_delivery_flag': MechanismPerformanceResult(), 'no_delivery': MechanismPerformanceResult(), 'cancellation': MechanismPerformanceResult(), 'is_dissatisfied': MechanismPerformanceResult()}, pnl_assumptions={'delivery_time': (np.float64(0.0003212925302498171), np.True_, 0.05), 'is_dissatisfied': (np.float64(0.0), np.True_, 0.05), 'freight_value': (np.float64(1.0), np.False_, 0.05), 'total_payment': (np.float64(1.0), np.False_, 0.05), 'delivery_delay': (np.float64(0.007922408560481209), np.True_, 0.05), 'late_delivery_flag': (np.float64(0.0), np.True_, 0.05), 'no_delivery': (np.float64(0.0

In [ ]:
# BASELINE SAMPLE
baseline=(data_cluster1["is_dissatisfied"].mean())
print("baseline_dissatisfaction")
baseline

baseline_dissatisfaction


np.float64(0.1233608776514349)

intervensi

In [ ]:
# INTERVENSI 1 (freight_value)
# TURUNKAN ONGKIR
int_freight = gcm.interventional_samples(
    causal_model,
    interventions={
        "freight_value": lambda x: x * 0.9
    },
    observed_data = data_cluster1
)

In [ ]:
print("is_dissatisfied intervention:")
print(data_cluster1["is_dissatisfied"].mean())

is_dissatisfied intervention:
0.1233608776514349


In [ ]:
print("after intervention")
print(int_freight["is_dissatisfied"].mean())

after intervention
0.12409068437036513


In [ ]:
comparison = pd.DataFrame({
    "Variabel": ["freight_value", "is_dissatisfied"],
    "Sebelum": [
        data_cluster1["freight_value"].mean(),
        data_cluster1["is_dissatisfied"].mean()
    ],
    "Sesudah": [
        int_freight["freight_value"].mean(),
        int_freight["is_dissatisfied"].mean()
    ]
})

comparison["Efek Intervensi"] = comparison["Sesudah"] - comparison["Sebelum"]

comparison

,Variabel,Sebelum,Sesudah,Efek Intervensi
0,freight_value,18.140062,16.353032,-1.78703
1,is_dissatisfied,0.123361,0.124091,0.00073


In [ ]:
# INTERVENSI 2
# PERCEPAT PENGIRIMAN
# delivery time turun 10%
int_delivery = gcm.interventional_samples(
    causal_model,
    interventions={
        "delivery_time": lambda x: x * 0.9
    },
    observed_data = data_cluster1
)

In [ ]:
print("is_dissatisfied intervention:")
print(data_cluster1["is_dissatisfied"].mean())

is_dissatisfied intervention:
0.1233608776514349


In [ ]:
print("after intervention")
print(int_delivery["is_dissatisfied"].mean())

after intervention
0.11836994138004096


In [ ]:
comparison = pd.DataFrame({
    "Variabel": ["delivery_time", "is_dissatisfied"],
    "Sebelum": [
        data_cluster1["delivery_time"].mean(),
        data_cluster1["is_dissatisfied"].mean()
    ],
    "Sesudah": [
        int_delivery["delivery_time"].mean(),
        int_delivery["is_dissatisfied"].mean()
    ]
})

comparison["Efek Intervensi"] = comparison["Sesudah"] - comparison["Sebelum"]

comparison

,Variabel,Sebelum,Sesudah,Efek Intervensi
0,delivery_time,11.318313,10.198432,-1.119881
1,is_dissatisfied,0.123361,0.118370,-0.004991


In [ ]:
# INTERVENSI 3 (late_delivery_flag)
int_late = gcm.interventional_samples(
    causal_model,
    interventions={
        "late_delivery_flag": lambda x: x * 0.9
    },
    observed_data = data_cluster1
)

In [ ]:
print("is_dissatisfied intervention:")
print(data_cluster1["is_dissatisfied"].mean())

is_dissatisfied intervention:
0.1233608776514349


In [ ]:
print("after intervention")
print(int_late["is_dissatisfied"].mean())

after intervention
0.12781034442168704


In [ ]:
comparison = pd.DataFrame({
    "Variabel": ["late_delivery_flag", "is_dissatisfied"],
    "Sebelum": [
        data_cluster1["late_delivery_flag"].mean(),
        data_cluster1["is_dissatisfied"].mean()
    ],
    "Sesudah": [
        int_late["late_delivery_flag"].mean(),
        int_late["is_dissatisfied"].mean()
    ]
})

comparison["Efek Intervensi"] = comparison["Sesudah"] - comparison["Sebelum"]

comparison

,Variabel,Sebelum,Sesudah,Efek Intervensi
0,late_delivery_flag,0.072141,0.064919,-0.007222
1,is_dissatisfied,0.123361,0.127810,0.004449


In [ ]:
# intervensi 4
# no delivery
int_no_delivery = gcm.interventional_samples(
    causal_model,
    interventions={
        "no_delivery": lambda x: 0
    },
    observed_data = data_cluster1
)

In [ ]:
print("is_dissatisfied intervention:")
print(data_cluster1["is_dissatisfied"].mean())

is_dissatisfied intervention:
0.1233608776514349


In [ ]:
print("after intervention")
print(int_no_delivery["is_dissatisfied"].mean())

after intervention
0.12225439649692775


In [ ]:
comparison = pd.DataFrame({
    "Variabel": ["no_delivery", "is_dissatisfied"],
    "Sebelum": [
        data_cluster1["no_delivery"].mean(),
        data_cluster1["is_dissatisfied"].mean()
    ],
    "Sesudah": [
        int_no_delivery["no_delivery"].mean(),
        int_no_delivery["is_dissatisfied"].mean()
    ]
})

comparison["Efek Intervensi"] = comparison["Sesudah"] - comparison["Sebelum"]

comparison

,Variabel,Sebelum,Sesudah,Efek Intervensi
0,no_delivery,0.000910,0.00091,0.000000
1,is_dissatisfied,0.123361,0.12781,0.004449


In [ ]:
# RINGKASAN
summary = pd.DataFrame({
    "Scenario": [
        "Baseline",
        "Freight Value ↓",
        "Delivery Time ↓",
        "late_delivery_flag ↓",
        "No delivery ↑",
    ],
    "Mean_is_dissatisfied": [
        baseline.mean(),
        int_freight["is_dissatisfied"].mean(),
        int_delivery["is_dissatisfied"].mean(),
        int_late["is_dissatisfied"].mean(),
        int_no_delivery["is_dissatisfied"].mean(),
    ]
})

display(summary)

,Scenario,Mean_is_dissatisfied
0,Baseline,0.123361
1,Freight Value ↓,0.124091
2,Delivery Time ↓,0.118370
3,late_delivery_flag ↓,0.127810
4,No delivery ↑,0.122254
